# Full XGBoost + KMeansSMOTE-Style Experiment Notebook

This notebook follows the original full XGBoost workflow:

- multiple seeds
- multiple resampling methods: `none`, `smote`, `kmeans`
- 10-fold CV for each setting
- probability files for each setting
- multiple fixed-threshold submission files
- multiple target-true-rate submission files
- `cv_scores.csv`
- `submission_summary.csv`

Important note: this notebook does **not** import `imblearn`, because some local environments have `imbalanced-learn / scikit-learn / pytest` compatibility conflicts.  
The official `.py` script in `src/` can keep the real `KMeansSMOTE`. This notebook uses local fallback implementations so it can run locally without environment issues.

Before running, place these files in the same folder as this notebook:

```text
train.csv
test.csv
sample_submission.csv
```


In [1]:
# Optional installation cell
# If xgboost is missing, uncomment and run this line, then restart the kernel.

# import sys
# !{sys.executable} -m pip install xgboost


In [2]:
# ============================================================
# 1. Import libraries
# ============================================================

from __future__ import annotations

from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.cluster import KMeans
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import OneHotEncoder
from sklearn.utils import shuffle

from xgboost import XGBClassifier

print("Libraries imported successfully.")


Libraries imported successfully.


In [3]:
# ============================================================
# 2. Local paths
# ============================================================

TRAIN_PATH = "train.csv"
TEST_PATH = "test.csv"
SAMPLE_SUBMISSION_PATH = "sample_submission.csv"

OUTPUT_DIR = Path("submissions/xgb_optuna_style_kmeans")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("TRAIN_PATH:", TRAIN_PATH)
print("TEST_PATH:", TEST_PATH)
print("SAMPLE_SUBMISSION_PATH:", SAMPLE_SUBMISSION_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)


TRAIN_PATH: train.csv
TEST_PATH: test.csv
SAMPLE_SUBMISSION_PATH: sample_submission.csv
OUTPUT_DIR: submissions\xgb_optuna_style_kmeans


In [4]:
# ============================================================
# 3. Configuration
# ============================================================

TARGET = "Transported"

EXPENSE_COLS = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]

NUM_COLS = [
    "ShoppingMall",
    "FoodCourt",
    "RoomService",
    "Spa",
    "VRDeck",
    "Expenses",
    "Age",
]

CAT_COLS = [
    "CryoSleep",
    "Cabin_1",
    "Cabin_3",
    "VIP",
    "HomePlanet",
    "Destination",
]

DROP_LIST = [
    "ShoppingMall",
    "Age",
    "CryoSleep_True",
    "HomePlanet_Earth",
    "HomePlanet_Europa",
    "VIP_True",
    "HomePlanet_Mars",
    "Destination_PSO J318.5-22",
    "VIP_False",
    "Destination_55 Cancri e",
    "FoodCourt",
    "Destination_TRAPPIST-1e",
]

XGB_PARAMS = {
    "reg_lambda": 3.0610042624477543,
    "reg_alpha": 4.581902571574289,
    "colsample_bytree": 0.9241969052729379,
    "subsample": 0.9527591724824661,
    "learning_rate": 0.06672065863100594,
    "n_estimators": 730,
    "max_depth": 5,
    "min_child_weight": 1,
    "num_parallel_tree": 1,
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "tree_method": "hist",
    "random_state": 42,
    "n_jobs": 4,
}

SEEDS = [42, 123, 2023, 2140]
METHODS = ["none", "smote", "kmeans"]
FIXED_THRESHOLDS = [0.50, 0.51, 0.525, 0.535, 0.545]
TARGET_TRUE_RATES = [0.4865, 0.488, 0.490, 0.492]

print("Configuration loaded.")


Configuration loaded.


In [5]:
# ============================================================
# 4. Load data
# ============================================================

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample = pd.read_csv(SAMPLE_SUBMISSION_PATH)

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Sample submission shape:", sample.shape)

display(train.head())


Train shape: (8693, 14)
Test shape: (4277, 13)
Sample submission shape: (4277, 2)


,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [6]:
# ============================================================
# 5. Preprocessing and feature engineering
# Same logic as the original XGBoost script
# ============================================================

def notebook_preprocess(
    train: pd.DataFrame,
    test: pd.DataFrame,
    seed: int,
) -> tuple[pd.DataFrame, pd.Series, pd.DataFrame]:
    full = pd.concat([train, test], ignore_index=True)

    full.loc[full["CryoSleep"].eq(True), EXPENSE_COLS] = 0
    full["Expenses"] = full[EXPENSE_COLS].sum(axis=1)
    full.loc[full["CryoSleep"].isna() & full["Expenses"].eq(0), "CryoSleep"] = True

    full["Name"] = full["Name"].fillna("Unknown Unknown")
    full["Room"] = full["PassengerId"].str[:4]

    for col in ["VIP", "Cabin", "HomePlanet", "Destination"]:
        guide = full[["Room", col]].dropna().drop_duplicates("Room")
        full = full.merge(guide, how="left", on="Room", suffixes=("", "_guide"))
        full[col] = full[col].fillna(full[f"{col}_guide"])
        full = full.drop(columns=[f"{col}_guide"])

    cabin = full["Cabin"].str.split("/", expand=True)
    full["Cabin_1"] = cabin[0]
    full["Cabin_2"] = cabin[1]
    full["Cabin_3"] = cabin[2]

    selected = full[NUM_COLS + CAT_COLS + [TARGET]].copy()

    num_imp = SimpleImputer(strategy="mean")
    cat_imp = SimpleImputer(strategy="most_frequent")

    selected[NUM_COLS] = pd.DataFrame(
        num_imp.fit_transform(selected[NUM_COLS]),
        columns=NUM_COLS,
    )

    selected[CAT_COLS] = pd.DataFrame(
        cat_imp.fit_transform(selected[CAT_COLS]),
        columns=CAT_COLS,
    )

    try:
        encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

    encoded = pd.DataFrame(
        encoder.fit_transform(selected[CAT_COLS]),
        columns=encoder.get_feature_names_out(CAT_COLS),
    )

    selected = pd.concat([selected.drop(columns=CAT_COLS), encoded], axis=1)

    train_processed = selected[selected[TARGET].notna()].copy()
    test_processed = selected[selected[TARGET].isna()].drop(columns=[TARGET]).copy()

    y = train_processed[TARGET].astype(int)
    X = train_processed.drop(columns=[TARGET])

    X, y = shuffle(X, y, random_state=seed)

    X = X.reset_index(drop=True)
    y = y.reset_index(drop=True)
    test_processed = test_processed.reset_index(drop=True)

    drop_cols = [col for col in DROP_LIST if col in X.columns]
    X = X.drop(columns=drop_cols)
    test_processed = test_processed.drop(
        columns=[col for col in drop_cols if col in test_processed.columns]
    )

    return X, y, test_processed


X_debug, y_debug, X_test_debug = notebook_preprocess(train, test, seed=42)

print("Processed train shape:", X_debug.shape)
print("Processed test shape:", X_test_debug.shape)
print("Target distribution:")
print(y_debug.value_counts())

display(X_debug.head())


Processed train shape: (8693, 15)
Processed test shape: (4277, 15)
Target distribution:
Transported
1    4378
0    4315
Name: count, dtype: int64


,RoomService,Spa,VRDeck,Expenses,CryoSleep_False,Cabin_1_A,Cabin_1_B,Cabin_1_C,Cabin_1_D,Cabin_1_E,Cabin_1_F,Cabin_1_G,Cabin_1_T,Cabin_3_P,Cabin_3_S
0,417.0,3.000000,1057.0,2460.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
1,4.0,0.000000,1.0,909.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
2,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
3,0.0,305.896819,0.0,774.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
4,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0


In [7]:
# ============================================================
# 6. Local fallback resampling
# Full experiment version without imblearn
# ============================================================

def simple_smote_fallback(
    X: pd.DataFrame,
    y: pd.Series,
    seed: int = 42,
    target_ratio: float = 1.0,
    k_neighbors: int = 5,
) -> tuple[pd.DataFrame, pd.Series]:
    rng = np.random.default_rng(seed)

    X_np = X.to_numpy(dtype=float)
    y_np = y.to_numpy(dtype=int)

    classes, counts = np.unique(y_np, return_counts=True)

    if len(classes) != 2:
        return X.copy(), y.copy()

    minority_class = classes[np.argmin(counts)]
    n_min = counts.min()
    n_maj = counts.max()
    target_min = int(n_maj * target_ratio)
    n_to_generate = target_min - n_min

    if n_to_generate <= 0:
        return X.copy(), y.copy()

    X_min = X_np[y_np == minority_class]

    if len(X_min) <= 1:
        return X.copy(), y.copy()

    n_neighbors = min(k_neighbors + 1, len(X_min))
    nn = NearestNeighbors(n_neighbors=n_neighbors)
    nn.fit(X_min)
    neighbors = nn.kneighbors(X_min, return_distance=False)

    synthetic_samples = []

    for _ in range(n_to_generate):
        idx = rng.integers(0, len(X_min))
        neighbor_candidates = neighbors[idx][1:]

        if len(neighbor_candidates) == 0:
            neighbor_idx = idx
        else:
            neighbor_idx = rng.choice(neighbor_candidates)

        x_i = X_min[idx]
        x_j = X_min[neighbor_idx]

        gap = rng.random()
        synthetic = x_i + gap * (x_j - x_i)
        synthetic_samples.append(synthetic)

    X_syn = np.vstack(synthetic_samples)
    y_syn = np.full(len(X_syn), minority_class)

    X_res = np.vstack([X_np, X_syn])
    y_res = np.concatenate([y_np, y_syn])

    X_res = pd.DataFrame(X_res, columns=X.columns)
    y_res = pd.Series(y_res, name=y.name)

    X_res, y_res = shuffle(X_res, y_res, random_state=seed)

    return X_res.reset_index(drop=True), y_res.reset_index(drop=True)


def kmeans_smote_style_fallback(
    X: pd.DataFrame,
    y: pd.Series,
    seed: int = 42,
    n_clusters: int = 8,
    k_neighbors: int = 5,
) -> tuple[pd.DataFrame, pd.Series]:
    rng = np.random.default_rng(seed)

    X_np = X.to_numpy(dtype=float)
    y_np = y.to_numpy(dtype=int)

    if len(np.unique(y_np)) != 2:
        return X.copy(), y.copy()

    n_clusters = min(n_clusters, max(2, len(X) // 500))
    n_clusters = max(2, n_clusters)

    try:
        kmeans = KMeans(
            n_clusters=n_clusters,
            random_state=seed,
            n_init=10,
        )
        cluster_labels = kmeans.fit_predict(X_np)
    except Exception as exc:
        print("KMeans failed; falling back to simple SMOTE. Reason:", exc)
        return simple_smote_fallback(X, y, seed=seed, k_neighbors=k_neighbors)

    synthetic_X_list = []
    synthetic_y_list = []

    for cluster_id in np.unique(cluster_labels):
        idx = np.where(cluster_labels == cluster_id)[0]

        if len(idx) < 10:
            continue

        X_cluster = X.iloc[idx].reset_index(drop=True)
        y_cluster = y.iloc[idx].reset_index(drop=True)

        counts = y_cluster.value_counts()

        if len(counts) < 2:
            continue

        min_count = counts.min()
        max_count = counts.max()

        if max_count - min_count < 3:
            continue

        X_bal, y_bal = simple_smote_fallback(
            X_cluster,
            y_cluster,
            seed=int(rng.integers(0, 1_000_000)),
            target_ratio=1.0,
            k_neighbors=min(k_neighbors, max(1, min_count - 1)),
        )

        n_original = len(X_cluster)

        if len(X_bal) > n_original:
            synthetic_X_list.append(X_bal.iloc[n_original:])
            synthetic_y_list.append(y_bal.iloc[n_original:])

    if not synthetic_X_list:
        print("KMeans-style fallback generated no samples; using simple SMOTE fallback.")
        return simple_smote_fallback(X, y, seed=seed, k_neighbors=k_neighbors)

    X_syn = pd.concat(synthetic_X_list, ignore_index=True)
    y_syn = pd.concat(synthetic_y_list, ignore_index=True)

    X_res = pd.concat([X.reset_index(drop=True), X_syn], ignore_index=True)
    y_res = pd.concat([y.reset_index(drop=True), y_syn], ignore_index=True)

    X_res, y_res = shuffle(X_res, y_res, random_state=seed)

    return X_res.reset_index(drop=True), y_res.reset_index(drop=True)


def resample(
    X: pd.DataFrame,
    y: pd.Series,
    method: str,
    seed: int,
) -> tuple[pd.DataFrame, pd.Series]:
    if method == "none":
        return X, y

    if method == "smote":
        return simple_smote_fallback(X, y, seed=seed)

    if method == "kmeans":
        return kmeans_smote_style_fallback(X, y, seed=seed)

    raise ValueError(f"Unknown method: {method}")


for method in METHODS:
    X_tmp, y_tmp = resample(X_debug, y_debug, method=method, seed=42)
    print(method, X_tmp.shape, y_tmp.value_counts().to_dict())


none (8693, 15) {1: 4378, 0: 4315}
smote (8756, 15) {0: 4378, 1: 4378}
kmeans (10741, 15) {1: 5389, 0: 5352}


In [8]:
# ============================================================
# 7. Evaluation and submission helper functions
# ============================================================

def cv_score(X: pd.DataFrame, y: pd.Series) -> tuple[float, float]:
    cv = StratifiedKFold(
        n_splits=10,
        shuffle=True,
        random_state=2140,
    )

    scores = cross_val_score(
        XGBClassifier(**XGB_PARAMS),
        X,
        y,
        scoring="accuracy",
        cv=cv,
        n_jobs=1,
    )

    return float(scores.mean()), float(scores.std())


def write_submission(
    path: Path,
    passenger_id: pd.Series,
    prob: np.ndarray,
    threshold: float,
) -> dict[str, object]:
    pred = prob >= threshold

    pd.DataFrame({
        "PassengerId": passenger_id,
        TARGET: pred.astype(bool),
    }).to_csv(path, index=False)

    return {
        "file": path.name,
        "threshold": threshold,
        "true_rate": float(pred.mean()),
    }


def threshold_for_true_rate(prob: np.ndarray, true_rate: float) -> float:
    return float(np.quantile(prob, 1.0 - true_rate))


print("Helper functions ready.")


Helper functions ready.


In [9]:
# ============================================================
# 8. Full experiment loop
# This follows the original script structure
# ============================================================

summary_rows = []
score_rows = []

for seed in SEEDS:
    print("\n" + "=" * 70)
    print(f"Seed = {seed}")
    print("=" * 70)

    X, y, X_test = notebook_preprocess(train, test, seed=seed)

    print("Base processed shape:", X.shape)
    print("Class distribution:", y.value_counts().to_dict())

    for method in METHODS:
        print("\n" + "-" * 50)
        print(f"Method = {method}")
        print("-" * 50)

        try:
            X_fit, y_fit = resample(X, y, method, seed)

            print("Training shape after resampling:", X_fit.shape)
            print("Class distribution after resampling:", y_fit.value_counts().to_dict())

            mean, std = cv_score(X_fit, y_fit)

            score_rows.append({
                "seed": seed,
                "method": method,
                "cv_mean": mean,
                "cv_std": std,
                "error": "",
            })

            print(f"CV accuracy = {mean:.5f} +/- {std:.5f}")

            model = XGBClassifier(**XGB_PARAMS)
            model.fit(X_fit, y_fit)

            prob = model.predict_proba(X_test)[:, 1]

            prob_path = OUTPUT_DIR / f"prob_seed{seed}_{method}.csv"

            pd.DataFrame({
                "PassengerId": sample["PassengerId"],
                "probability": prob,
            }).to_csv(prob_path, index=False)

            print("Saved probability file:", prob_path.name)

            for threshold in FIXED_THRESHOLDS:
                result = write_submission(
                    OUTPUT_DIR / f"submission_xgb_optuna_seed{seed}_{method}_thr{int(threshold * 1000):03d}.csv",
                    sample["PassengerId"],
                    prob,
                    threshold,
                )

                result.update({
                    "seed": seed,
                    "method": method,
                    "type": "fixed_threshold",
                })

                summary_rows.append(result)

            for true_rate in TARGET_TRUE_RATES:
                threshold = threshold_for_true_rate(prob, true_rate)

                result = write_submission(
                    OUTPUT_DIR / f"submission_xgb_optuna_seed{seed}_{method}_rate{int(true_rate * 10000):04d}.csv",
                    sample["PassengerId"],
                    prob,
                    threshold,
                )

                result.update({
                    "seed": seed,
                    "method": method,
                    "type": "target_true_rate",
                    "target_true_rate": true_rate,
                })

                summary_rows.append(result)

        except Exception as exc:
            print(f"FAILED: seed={seed}, method={method}, error={exc}")

            score_rows.append({
                "seed": seed,
                "method": method,
                "cv_mean": np.nan,
                "cv_std": np.nan,
                "error": str(exc),
            })



Seed = 42
Base processed shape: (8693, 15)
Class distribution: {1: 4378, 0: 4315}

--------------------------------------------------
Method = none
--------------------------------------------------
Training shape after resampling: (8693, 15)
Class distribution after resampling: {1: 4378, 0: 4315}
CV accuracy = 0.80904 +/- 0.01349
Saved probability file: prob_seed42_none.csv

--------------------------------------------------
Method = smote
--------------------------------------------------
Training shape after resampling: (8756, 15)
Class distribution after resampling: {0: 4378, 1: 4378}
CV accuracy = 0.80711 +/- 0.01682
Saved probability file: prob_seed42_smote.csv

--------------------------------------------------
Method = kmeans
--------------------------------------------------
Training shape after resampling: (10741, 15)
Class distribution after resampling: {1: 5389, 0: 5352}
CV accuracy = 0.81892 +/- 0.01032
Saved probability file: prob_seed42_kmeans.csv

Seed = 123
Base proce

In [ ]:
# ============================================================
# 9. Save summaries
# ============================================================

cv_scores = pd.DataFrame(score_rows)
submission_summary = pd.DataFrame(summary_rows)

cv_scores.to_csv(OUTPUT_DIR / "cv_scores.csv", index=False)
submission_summary.to_csv(OUTPUT_DIR / "submission_summary.csv", index=False)

print("\n" + "=" * 70)
print("All done.")
print("Saved outputs to:", OUTPUT_DIR)
print("=" * 70)

print("\nCV scores:")
display(cv_scores)

print("\nSubmission summary:")
display(submission_summary.head(30))

print("\nGenerated files:")
for file in sorted(OUTPUT_DIR.glob("*.csv"))[:40]:
    print(file.name)

print("\nIf there are many files, check this folder manually:")
print(OUTPUT_DIR.resolve())


## Notes

This notebook follows the original multi-seed and multi-threshold experiment structure.  
Because it avoids `imblearn`, the `kmeans` method uses a local KMeans-SMOTE-style fallback.  
For the official real `KMeansSMOTE` implementation, use the Python script in `src/train_XGBoost_KMeansSMOTE.py`.
